In [2]:
"""Stress Testing & Liquidity Tool (single file). Run: python stress_tool.py
Needs: pip install numpy pandas   (Python 3.10+)"""
import os
import numpy as np
import pandas as pd

# ===== portfolio =====
HOLDINGS = pd.DataFrame([
    # name,              class,       mv,  duration, hqla
    ("Cash at central bank", "cash",     60, 0.0, "L1"),
    ("US Treasury 10Y",      "gov",     120, 8.0, "L1"),
    ("US Treasury 2Y",       "gov",     100, 1.9, "L1"),
    ("IG corporate bonds",   "corp",     50, 5.0, "L2A"),
    ("Large-cap equities",   "equity",   25, 0.0, "L2B"),
    ("HY corporate bonds",   "corp",     40, 4.0, None),
    ("EM equities",          "equity",   30, 0.0, None),
    ("Gold",                 "commodity", 25, 0.0, None),
    ("EUR/USD position",     "fx",       35, 0.0, None),
], columns=["name", "asset_class", "market_value", "duration", "hqla"])


# ===== scenarios =====
HISTORICAL = {
    "GFC 2008 (Sep-Nov)":   dict(equity=-0.35, rate_bp=-100, spread_bp=+400, fx=-0.08, commodity=-0.15, outflow_mult=1.50),
    "Covid crash Mar 2020": dict(equity=-0.30, rate_bp=-60,  spread_bp=+250, fx=-0.03, commodity=-0.05, outflow_mult=1.30),
    "2022 rates shock":     dict(equity=-0.20, rate_bp=+250, spread_bp=+100, fx=-0.10, commodity=-0.02, outflow_mult=1.10),
    "Black Monday 1987":    dict(equity=-0.22, rate_bp=-30,  spread_bp=+50,  fx=-0.02, commodity=-0.03, outflow_mult=1.10),
}
HYPOTHETICAL = {
    "Rates +200bp":            dict(rate_bp=+200),
    "Equity -30%":             dict(equity=-0.30),
    "Credit spreads +300bp":   dict(spread_bp=+300),
    "USD rally (FX -15%)":     dict(fx=-0.15),
    "Combined severe":         dict(equity=-0.40, rate_bp=+150, spread_bp=+350, fx=-0.12, commodity=-0.20, outflow_mult=1.75),
}
ALL_SCENARIOS = {**HISTORICAL, **HYPOTHETICAL}
DEFAULTS = dict(equity=0.0, rate_bp=0.0, spread_bp=0.0, fx=0.0, commodity=0.0, outflow_mult=1.0)


def full(shock: dict) -> dict:
    return {**DEFAULTS, **shock}


def position_pnl(holdings: pd.DataFrame, shock: dict) -> pd.Series:
    s = full(shock)
    pnl = []
    for _, r in holdings.iterrows():
        mv, dur, c = r.market_value, r.duration, r.asset_class
        if c == "equity":
            p = mv * s["equity"]
        elif c == "gov":
            p = -mv * dur * s["rate_bp"] / 1e4
        elif c == "corp":
            p = -mv * dur * (s["rate_bp"] + s["spread_bp"]) / 1e4
        elif c == "fx":
            p = mv * s["fx"]
        elif c == "commodity":
            p = mv * s["commodity"]
        else:
            p = 0.0
        pnl.append(p)
    return pd.Series(pnl, index=holdings.name)


def run_scenarios(holdings, scenarios=ALL_SCENARIOS) -> pd.DataFrame:
    table = pd.DataFrame({k: position_pnl(holdings, v) for k, v in scenarios.items()})
    table.loc["TOTAL P&L"] = table.sum()
    total_mv = holdings.market_value.sum()
    table.loc["% of portfolio"] = table.loc["TOTAL P&L"] / total_mv * 100
    return table.round(2)


# ===== liquidity =====
HAIRCUT = {"L1": 0.00, "L2A": 0.15, "L2B": 0.50}

OUTFLOWS = pd.DataFrame([
    ("Retail stable deposits",           800, 0.05),
    ("Retail less-stable deposits",      300, 0.10),
    ("Operational wholesale deposits",   400, 0.25),
    ("Non-operational wholesale funding", 500, 0.40),
    ("Undrawn credit/liquidity lines",   300, 0.10),
    ("Derivative collateral calls",       80, 1.00),
], columns=["item", "balance", "runoff"])

INFLOWS = pd.DataFrame([
    ("Retail loan repayments",      200, 0.50),
    ("Wholesale performing loans",  250, 0.50),
], columns=["item", "balance", "inflow_rate"])

# Contractual cash-flow ladder: (bucket, inflows, outflows, cumulative gap limit)
LADDER = pd.DataFrame([
    ("Overnight",   20,  60, -40),
    ("2-7 days",    35,  70, -80),
    ("8-30 days",   90, 130, -120),
    ("31-90 days", 150, 170, -130),
    ("91-180 days", 160, 150, -130),
    ("181-365 days", 220, 190, -110),
], columns=["bucket", "inflows", "outflows", "cum_gap_limit"])

INTERNAL_LCR_LIMIT = 1.10      # internal early-warning limit
REGULATORY_LCR_MIN = 1.00


def hqla_amount(holdings: pd.DataFrame, shock: dict | None = None):
    """Stock of HQLA after haircuts and Basel L2/L2B caps; revalued under `shock` if given."""
    h = holdings[holdings.hqla.notna()].copy()
    mv = h.market_value.copy()
    if shock:
        mv = mv + position_pnl(holdings, shock).loc[h.name].to_numpy()
    h["value"] = mv.clip(lower=0).to_numpy() * (1 - h.hqla.map(HAIRCUT).to_numpy())
    L1 = h.loc[h.hqla == "L1", "value"].sum()
    L2A = h.loc[h.hqla == "L2A", "value"].sum()
    L2B = h.loc[h.hqla == "L2B", "value"].sum()
    adj15 = max(L2B - 15 / 85 * (L1 + L2A), L2B - 15 / 60 * L1, 0)
    adj40 = max(L2A + L2B - adj15 - 2 / 3 * L1, 0)
    return L1 + L2A + L2B - adj15 - adj40, dict(L1=L1, L2A=L2A, L2B=L2B)


def lcr(holdings, shock: dict | None = None):
    mult = full(shock or {})["outflow_mult"]
    hqla, parts = hqla_amount(holdings, shock)
    out = (OUTFLOWS.balance * (OUTFLOWS.runoff * mult).clip(upper=1.0)).sum()
    inflow = (INFLOWS.balance * INFLOWS.inflow_rate).sum()
    net = out - min(inflow, 0.75 * out)               # inflows capped at 75% of outflows
    ratio = hqla / net
    status = ("BREACH - below regulatory minimum" if ratio < REGULATORY_LCR_MIN
              else "WARNING - below internal limit" if ratio < INTERNAL_LCR_LIMIT else "OK")
    return dict(HQLA=round(hqla, 1), Outflows=round(out, 1), Inflows_counted=round(min(inflow, 0.75 * out), 1),
                Net_outflows=round(net, 1), LCR=round(ratio, 3), Status=status)


def liquidity_gap(ladder=LADDER) -> pd.DataFrame:
    d = ladder.copy()
    d["gap"] = d.inflows - d.outflows
    d["cum_gap"] = d.gap.cumsum()
    d["limit_breach"] = d.cum_gap < d.cum_gap_limit
    d["flag"] = d.limit_breach.map({True: "BREACH", False: "ok"})
    return d.drop(columns="limit_breach")


def stressed_lcr_table(holdings, scenarios) -> pd.DataFrame:
    rows = {name: lcr(holdings, shock) for name, shock in scenarios.items()}
    rows = {"Base (no shock)": lcr(holdings), **rows}
    return pd.DataFrame(rows).T


# ===== run =====
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
os.makedirs("results", exist_ok=True)

print(f"Portfolio market value: {HOLDINGS.market_value.sum():,.0f}m\n")

print("=== Historical scenarios: P&L (m) ===")
hist = run_scenarios(HOLDINGS, HISTORICAL)
print(hist.to_string())
print("\n=== Hypothetical scenarios: P&L (m) ===")
hyp = run_scenarios(HOLDINGS, HYPOTHETICAL)
print(hyp.to_string())
pd.concat([hist, hyp], axis=1).to_csv("results/stress_pnl.csv")

print("\n=== Base LCR ===")
for k, v in lcr(HOLDINGS).items():
    print(f"{k:>16}: {v}")

print("\n=== Liquidity gap ladder (limit-breach flagging) ===")
gap = liquidity_gap()
print(gap.to_string(index=False))
gap.to_csv("results/liquidity_gap.csv", index=False)

print("\n=== LCR under stress scenarios (HQLA revalued, runoff scaled) ===")
tbl = stressed_lcr_table(HOLDINGS, ALL_SCENARIOS)
print(tbl.to_string())
tbl.to_csv("results/stressed_lcr.csv")
print("\nSaved CSVs to results/")


Portfolio market value: 485m

=== Historical scenarios: P&L (m) ===
                      GFC 2008 (Sep-Nov)  Covid crash Mar 2020  2022 rates shock  Black Monday 1987
name                                                                                               
Cash at central bank                0.00                  0.00              0.00               0.00
US Treasury 10Y                     9.60                  5.76            -24.00               2.88
US Treasury 2Y                      1.90                  1.14             -4.75               0.57
IG corporate bonds                 -7.50                 -4.75             -8.75              -0.50
Large-cap equities                 -8.75                 -7.50             -5.00              -5.50
HY corporate bonds                 -4.80                 -3.04             -5.60              -0.32
EM equities                       -10.50                 -9.00             -6.00              -6.60
Gold                            